# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [10]:
# constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

API key looks good so far


In [ ]:
# set up environment

In [3]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [11]:
# Get llama3.2 to answer, with streaming

def stream_programming_answer(client, model, question):
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": '''You are a programming wiz and a very good thorough tutor. 
            You answer patiently any programming question.
            '''},
            {"role": "user", "content": question}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [12]:
# Get Llama 3.2 to answer
stream_programming_answer(ollama_via_openai, MODEL_LLAMA, question)

**Decoding the Yield From Statement**

Let's break down this line of code:

```python
yield from {book.get("author") for book in books if book.get("author")}
```

This is a Python expression that uses the `yield from` syntax. To understand its behavior, we need to dive into what `yield from` does.

**What is Yield From?**

`yield from` is used to iterate over the results of another iterable or generator expression. It's equivalent to using a `for` loop to consume those results.

**Annotated Code with Comments**

To make it clearer, let's add comments and variable names:
```python
author_titles = {}
for book in books:  # Iterate over the 'books' collection
    if book.get("author"):  # Check if the book has an author
        title = book.get("title")  # Extract the book name (assuming it exists)
        author_titles[title] = title  # Store the title and author in a dictionary

# Yield from the dictionary directly
yield from {title: f"'{book.get('author')} - {book.get('title')}'" for title, authors in author_titles.items() if len(authors) > 1}
```
Note how we're creating a `for` loop that iterates over the books and extracts their titles. However, to avoid having an intermediate data structure (in this case, the `author_titles` dictionary), we use the `yield from` statement with a dictionary comprehension.

**How it Works**

Here's what happens when we execute this line:

1. The code generates a sequence of authors and titles using a dictionary comprehension within `{}`.
2. Each book that meets the condition (i.e., `book.get("author")`) is processed.
3. Yield from `title: f"'{book.get('author')} - {book.get('title')}'"**: We're effectively collecting key-value pairs where the key is a book title and the value is a string representing the author and book title.
4. Since this sequence yields individual items, we can consume them on-the-fly as they become available.

In summary, `yield from` allows us to iterate over a collection (in this case, a dictionary) using a more Pythonic and flexible syntax.